# SDT Model Analysis & Visualization

> Load a pre-trained **Soft Decision Tree (SDT)** on BloodMNIST with 70 haematology concept features.  
> Perform interpretability analysis: leaf/internal-node statistics, decision-path visualization,
> node-weight heat-vectors, top/bottom activation images, and batch export for paper figures.

**Checkpoint**: `./checkpoints/sdt_bloodmnist_concept70.pt`  
**Concept scores**: `./data/concept_70/`

## 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
from PIL import Image
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader, ConcatDataset
import medmnist
from medmnist import INFO
from torchvision import transforms

from SDT_pt_function import load_checkpoint_create
from sdt_visualization import (
    get_leaf_distribution, compute_internal_node_counts,
    visualize_sdt, build_concept_colors, plot_sample_with_concepts,
    DEFAULT_CATEGORY_COLORS,
    plot_node_heatvector, batch_export_node_heatvectors,
    compute_node_logits_for_dataset,
    visualize_top_k_images_for_node,
    visualize_top_and_bottom_k_images_for_node,
    analyze_all_nodes_summary,
)

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('cpu')
print('device:', device)

## 1. Concept Feature Metadata & Label Mapping

In [ ]:
cell_feature_metadata = {
    "nuclear_morphology_nc": {
        "en": "Nuclear morphology and N:C ratio",
        "zh": "细胞核形态与核浆结构",
        "features": [
            {"en": "Segmented nucleus", "zh": "分叶细胞核", "level": 4},
            {"en": "Band nucleus (band form)", "zh": "带状细胞核（未分叶）", "level": 4},
            {"en": "Bilobed nucleus", "zh": "双叶细胞核", "level": 4},
            {"en": "Round nucleus", "zh": "圆形细胞核", "level": 4},
            {"en": "Reniform / indented nucleus", "zh": "肾形/凹陷细胞核", "level": 4},
            {"en": "Coarse chromatin pattern", "zh": "染色质粗糙/团块状", "level": 4},
            {"en": "Prominent nucleoli", "zh": "明显核仁", "level": 4},
            {"en": "Irregular nuclear membrane", "zh": "不规则核膜", "level": 4},
            {"en": "Eccentric nucleus", "zh": "偏心细胞核", "level": 4},
            {"en": "Binucleated cell", "zh": "双核细胞", "level": 4},
            {"en": "Multinucleated cell", "zh": "多核细胞", "level": 4},
            {"en": "Clefted / notched nucleus", "zh": "裂隙/切迹样细胞核", "level": 4},
            {"en": "Oval nucleus", "zh": "椭圆形细胞核", "level": 4},
            {"en": "Pyknotic nucleus", "zh": "固缩细胞核", "level": 4},
            {"en": "Apoptotic nuclear fragments", "zh": "凋亡样核碎片", "level": 4},
            {"en": "Hypersegmented nucleus", "zh": "过度分叶细胞核", "level": 4},
            {"en": "Fine/open chromatin pattern", "zh": "染色质细致/疏松", "level": 4},
            {"en": "High nuclear-to-cytoplasmic ratio", "zh": "高核浆比", "level": 4},
            {"en": "Nuclear blebs / micronuclei", "zh": "核芽/微核", "level": 4},
            {"en": "Marked nuclear size variation (anisokaryosis)", "zh": "明显核大小不一（核异大）", "level": 4},
        ],
    },
    "cytoplasmic_tone_texture": {
        "en": "Cytoplasmic tone and texture",
        "zh": "胞质色调与质地",
        "features": [
            {"en": "Basophilic cytoplasm", "zh": "嗜碱性胞质", "level": 4},
            {"en": "Pale cytoplasm", "zh": "淡染胞质", "level": 4},
            {"en": "Cytoplasmic vacuoles", "zh": "胞质空泡", "level": 4},
            {"en": "Ground-glass cytoplasm", "zh": "毛玻璃样胞质", "level": 4},
            {"en": "Foamy cytoplasm", "zh": "泡沫样胞质", "level": 4},
            {"en": "Cytoplasmic blebs / projections", "zh": "胞质突起/胞质芽", "level": 4},
            {"en": "Perinuclear hof (Golgi zone clearing)", "zh": "核周晕（高尔基区透亮带）", "level": 4},
            {"en": "Polarized cytoplasm (one-sided)", "zh": "胞质极化（单侧胞质集中）", "level": 4},
            {"en": "Homogeneous dense cytoplasm", "zh": "均质致密胞质", "level": 4},
            {"en": "Paranuclear cytoplasmic vacuoles", "zh": "核周胞质空泡", "level": 4},
            {"en": "Clear cytoplasm", "zh": "透明样胞质", "level": 4},
            {"en": "Plasmacytoid cytoplasm", "zh": "浆细胞样胞质", "level": 4},
            {"en": "Peripheral basophilic rim", "zh": "细胞周边嗜碱性环", "level": 4},
        ],
    },
    "cytoplasmic_granules": {
        "en": "Cytoplasmic granules and inclusions",
        "zh": "胞质颗粒与内含物",
        "features": [
            {"en": "Fine azurophilic granules", "zh": "细嗜天青颗粒", "level": 4},
            {"en": "Eosinophilic granules", "zh": "嗜酸性颗粒（橙红）", "level": 4},
            {"en": "Basophilic granules", "zh": "嗜碱性颗粒（深紫）", "level": 4},
            {"en": "Toxic granulation", "zh": "中毒性颗粒", "level": 4},
            {"en": "Dohle bodies", "zh": "D\u00f6hle小体", "level": 4},
            {"en": "Auer rods", "zh": "Auer小体", "level": 4},
            {"en": "Granules obscure nucleus", "zh": "颗粒掩盖细胞核", "level": 4},
            {"en": "Hypogranular / agranular cytoplasm", "zh": "颗粒减少/无颗粒胞质", "level": 4},
            {"en": "Coarse cytoplasmic granules", "zh": "粗大胞质颗粒", "level": 4},
            {"en": "Giant cytoplasmic granules", "zh": "巨型胞质颗粒", "level": 4},
            {"en": "Azurophilic granules in lymphocytes", "zh": "淋巴细胞嗜天青颗粒", "level": 4},
            {"en": "Cytoplasmic crystalline inclusions", "zh": "晶样胞质内含物", "level": 4},
            {"en": "Cytoplasmic pigment granules", "zh": "色素颗粒（如含铁颗粒）", "level": 4},
            {"en": "Phagocytosed cellular debris", "zh": "吞噬细胞碎片/红细胞", "level": 4},
            {"en": "Basophilic stippling of erythrocytes", "zh": "红细胞嗜碱性点彩", "level": 4},
            {"en": "Howell\u2013Jolly bodies in erythrocytes", "zh": "红细胞内 Howell\u2013Jolly 小体", "level": 4},
            {"en": "Pappenheimer bodies in erythrocytes", "zh": "红细胞内 Pappenheimer 小体", "level": 4},
            {"en": "Uneven granule size (anisogranularity)", "zh": "颗粒大小不均（颗粒异大）", "level": 4},
            {"en": "Polar aggregation of cytoplasmic granules", "zh": "胞质颗粒向一极聚集", "level": 4},
            {"en": "Perinuclear sparing of cytoplasmic granules", "zh": "核周颗粒缺如带", "level": 4},
        ],
    },
    "non_leukocyte_elements": {
        "en": "Non-leukocyte blood elements",
        "zh": "非白细胞血细胞要素",
        "features": [
            {"en": "Nucleated erythrocyte (erythroblast)", "zh": "有核红细胞（幼红细胞）", "level": 4},
            {"en": "Platelet fragments / clumps", "zh": "血小板碎片/成团", "level": 4},
            {"en": "Red blood cell central pallor", "zh": "红细胞中央淡染区", "level": 4},
            {"en": "Red blood cell fragments (schistocytes)", "zh": "红细胞碎片（裂红细胞）", "level": 4},
            {"en": "Spherocytic red blood cell", "zh": "球形红细胞（无中央淡染）", "level": 4},
            {"en": "Target red blood cell (codocyte)", "zh": "靶形红细胞", "level": 4},
            {"en": "Teardrop red blood cell (dacrocyte)", "zh": "泪滴形红细胞", "level": 4},
            {"en": "Red blood cell anisocytosis (size variation)", "zh": "红细胞大小不一（红细胞异大）", "level": 4},
            {"en": "Red blood cell poikilocytosis (shape variation)", "zh": "红细胞形态多样（红细胞异形）", "level": 4},
            {"en": "Microcytic red blood cells", "zh": "小红细胞", "level": 4},
            {"en": "Macrocytic red blood cells", "zh": "大红细胞", "level": 4},
            {"en": "Rouleaux formation of red blood cells", "zh": "红细胞缗钱状排列", "level": 4},
            {"en": "Giant platelets", "zh": "巨大血小板", "level": 4},
        ],
    },
    "artifacts_quality": {
        "en": "Preparation and technical artifacts",
        "zh": "制片与技术伪影",
        "features": [
            {"en": "Stain precipitate (artifact)", "zh": "染色沉淀（伪影）", "level": 4},
            {"en": "Overlapping cell clumps (artifact)", "zh": "细胞重叠/成团（伪影）", "level": 4},
            {"en": "Out-of-focus (artifact)", "zh": "失焦（伪影）", "level": 4},
            {"en": "Motion blur (artifact)", "zh": "运动模糊（伪影）", "level": 4},
        ],
    },
}

def get_cell_features(metadata: dict, level: int = 1):
    """Flatten cell_feature_metadata into a list where feature['level'] <= level."""
    feats = []
    for cat in metadata.values():
        for f in cat.get('features', []):
            if f.get('level', 1) <= level:
                feats.append({'en': f['en'], 'zh': f['zh'], 'level': f['level']})
    return feats

cell_features = get_cell_features(cell_feature_metadata, level=4)
concept_list = [f['en'] for f in cell_features]
print(f'Total concepts: {len(concept_list)}')

# 8-class BloodMNIST labels
label_list = [
    'basophil', 'eosinophil', 'erythroblast', 'immature granulocytes',
    'lymphocyte', 'monocyte', 'neutrophil', 'platelet',
]
label_dict = {i: name for i, name in enumerate(label_list)}

concept_colors = build_concept_colors(cell_feature_metadata)
category_colors = DEFAULT_CATEGORY_COLORS

## 2. Load Dataset & Preprocessing

In [ ]:
# ---- Concept score CSV ----
OUT70 = Path('./data/concept_70')
train_df = pd.read_csv(OUT70 / 'concept70_logits_train.csv')
val_df   = pd.read_csv(OUT70 / 'concept70_logits_val.csv')
test_df  = pd.read_csv(OUT70 / 'concept70_logits_test.csv')

trainval_df = pd.concat([train_df, val_df], ignore_index=True)
feature_cols = [c for c in trainval_df.columns if c not in ['id', 'label', 'split']]

scaler = StandardScaler()
trainval_df[feature_cols] = scaler.fit_transform(trainval_df[feature_cols])
test_df[feature_cols]     = scaler.transform(test_df[feature_cols])

X_train = trainval_df[feature_cols].to_numpy(dtype=np.float32)
y_train = trainval_df['label'].to_numpy(dtype=np.int64)
X_test  = test_df[feature_cols].to_numpy(dtype=np.float32)
y_test  = test_df['label'].to_numpy(dtype=np.int64)

# ---- DataLoader ----
train_ds    = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
test_ds     = TensorDataset(torch.from_numpy(X_test),  torch.from_numpy(y_test))
trainloader = DataLoader(train_ds, batch_size=512, shuffle=False)
testloader  = DataLoader(test_ds,  batch_size=512)

# ---- BloodMNIST raw images ----
data_flag = 'bloodmnist'
DataClass = getattr(medmnist, INFO[data_flag]['python_class'])
common_tf = transforms.Compose([transforms.ToTensor()])

ds_train_raw = DataClass(split='train', size=224, as_rgb=True, mmap_mode='r', transform=common_tf, download=True)
ds_valid_raw = DataClass(split='val',   size=224, as_rgb=True, mmap_mode='r', transform=common_tf, download=True)
ds_test      = DataClass(split='test',  size=224, as_rgb=True, mmap_mode='r', transform=common_tf, download=True)
ds_train_img = ConcatDataset([ds_train_raw, ds_valid_raw])

print(f'X_train: {X_train.shape}  X_test: {X_test.shape}')
print(f'ds_train_img: {len(ds_train_img)}  ds_test: {len(ds_test)}')

## 3. Load SDT Checkpoint & Evaluate Test Accuracy

In [ ]:
ckpt_path = './checkpoints/sdt_bloodmnist_concept70.pt'
tree, optimizer, extra = load_checkpoint_create(ckpt_path, use_cuda=(device.type == 'cuda'))
print('extra:', extra)

# Evaluate on test set
tree.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device)
        pred = tree.forward(data).argmax(dim=1)
        all_preds.append(pred.cpu())
        all_targets.append(target.cpu())

all_preds   = torch.cat(all_preds)
all_targets = torch.cat(all_targets)
test_acc = (all_preds == all_targets).float().mean().item()
print(f'Test Accuracy: {test_acc*100:.3f}%  ({(all_preds==all_targets).sum().item()}/{len(all_targets)})')

## 4. Leaf Node Sample Distribution

In [ ]:
train_leaf_counts, train_leaf_class_counts, leaf_predictions = get_leaf_distribution(tree, trainloader, device)
test_leaf_counts,  test_leaf_class_counts,  _                = get_leaf_distribution(tree, testloader,  device)

L = tree.leaf_node_num_
print(f'Leaf nodes: {L}  |  Train samples: {train_leaf_counts.sum()}  |  Test samples: {test_leaf_counts.sum()}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))
x = np.arange(L)
w = 0.35

# --- Train vs Test bar chart ---
ax1 = axes[0]
b1 = ax1.bar(x - w/2, train_leaf_counts, w, label='Train', color='steelblue', alpha=0.8)
b2 = ax1.bar(x + w/2, test_leaf_counts,  w, label='Test',  color='coral',     alpha=0.8)
ax1.set(xlabel='Leaf Node Index', ylabel='Sample Count',
        title='Sample Distribution per Leaf Node (Train vs Test)')
ax1.set_xticks(x); ax1.set_xticklabels([f'L{i}' for i in range(L)], fontsize=8)
ax1.legend(); ax1.grid(axis='y', alpha=0.3)
for bars in (b1, b2):
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax1.annotate(f'{int(h)}', xy=(bar.get_x()+bar.get_width()/2, h),
                         xytext=(0,3), textcoords='offset points',
                         ha='center', va='bottom', fontsize=7, rotation=45)

# --- Per-class stacked bar (train) ---
ax2 = axes[1]
colors = plt.cm.tab10(np.linspace(0, 1, len(label_list)))
bottom = np.zeros(L)
for ci, cn in enumerate(label_list):
    cc = np.array([train_leaf_class_counts[l][ci] for l in range(L)])
    ax2.bar(x, cc, 0.7, bottom=bottom, label=cn, color=colors[ci], alpha=0.85)
    bottom += cc
ax2.set(xlabel='Leaf Node Index', ylabel='Sample Count',
        title='Per-Class Distribution per Leaf Node (Train)')
ax2.set_xticks(x)
ax2.set_xticklabels([f'L{i}\n({label_list[leaf_predictions[i]][:4]})' for i in range(L)], fontsize=7)
ax2.legend(loc='upper right', fontsize=8, ncol=2); ax2.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Detailed statistics table
print(f"{'Leaf':<6} {'Predicted Class':<25} {'Train':>7} {'Test':>7} {'Train%':>8} {'Test%':>8}")
print('-' * 65)
for l in range(L):
    pc = label_list[leaf_predictions[l]]
    tr, te = train_leaf_counts[l], test_leaf_counts[l]
    print(f'L{l:<5} {pc:<25} {tr:>7} {te:>7} {tr/train_leaf_counts.sum()*100:>7.2f}% {te/test_leaf_counts.sum()*100:>7.2f}%')
print('-' * 65)
print(f"{'Total':<32} {train_leaf_counts.sum():>7} {test_leaf_counts.sum():>7}")
print(f'Active leaves: Train {np.sum(train_leaf_counts>0)}/{L}, Test {np.sum(test_leaf_counts>0)}/{L}')

## 5. Internal Node Sample Flow

In [ ]:
train_internal_counts = compute_internal_node_counts(train_leaf_counts, tree.depth)
test_internal_counts  = compute_internal_node_counts(test_leaf_counts,  tree.depth)

print(f"{'Node':<6} {'Train':>10} {'Test':>10}")
print('-' * 28)
for i in range(len(train_internal_counts)):
    print(f'IN_{i:<3} {train_internal_counts[i]:>10} {test_internal_counts[i]:>10}')

## 6. SDT Decision-Path Visualization (Single Sample)

In [ ]:
# Pick a test sample and show its concept activation + SDT decision path
n = 75

sample, img, label, fig, axes = plot_sample_with_concepts(
    n=n, concept_dataset=test_ds, image_dataset=ds_test,
    concept_list=concept_list, label_dict=label_dict,
    cell_feature_metadata=cell_feature_metadata,
    concept_colors=concept_colors, figsize=(16, 6),
)
plt.show()
print(f'Label: {label} ({label_dict[label]})')

In [ ]:
fig, ax, info = visualize_sdt(tree, sample, figsize=(12, 7), title='SDT on one BloodMNIST sample')
plt.show()

node0 = info['internal_nodes'][0]
print(f'Root node \u2014 layer={node0["layer"]}, b={node0["b"]:.4f}, W-norm={node0["W"].norm():.4f}')
leaf0 = info['leaves'][0]
print('Leaf 0 top-3 probs:', torch.topk(leaf0['class_probs'], k=3))

## 7. Internal Node Weight Heat-Vector

In [ ]:
node_idx = 0
fig, ax = plot_node_heatvector(
    node_idx=node_idx, info=info, concept_list=concept_list,
    cell_feature_metadata=cell_feature_metadata,
    concept_colors=concept_colors, show=True,
)

## 8. Batch Export Weight Heat-Vectors & CSV

In [ ]:
output_dir = './analysis/tree_nodes'

exported_files = batch_export_node_heatvectors(
    info=info, concept_list=concept_list, output_dir=output_dir,
    cell_feature_metadata=cell_feature_metadata,
    concept_colors=concept_colors,
    cmap='Reds', figsize=(14, 4),
    node_indices=None, export_csv=True, csv_threshold=0.5,
)

## 9. Compute Node Logits on Training Set

In [ ]:
node_logits_train = compute_node_logits_for_dataset(tree, X_train, device)
print(f'Node logits shape: {node_logits_train.shape}  (samples x internal_nodes)')

## 10. Top-K / Bottom-K Activation Images per Node

In [ ]:
node_idx = 0  # change to inspect other nodes

# Top-20 highest activation images
fig, top_indices, top_scores = visualize_top_k_images_for_node(
    node_idx=node_idx, node_logits=node_logits_train,
    ds_images=ds_train_img, y_labels=y_train,
    label_dict=label_dict, tree_info=info, k=20, figsize=(18, 14),
)
plt.show()
print(f'Node {node_idx} Top-20 labels:', Counter(label_dict[y_train[i]] for i in top_indices).most_common())

In [ ]:
# High vs Low activation comparison (Top-10 each)
fig, top_idx, bot_idx = visualize_top_and_bottom_k_images_for_node(
    node_idx=node_idx, node_logits=node_logits_train,
    ds_images=ds_train_img, y_labels=y_train,
    label_dict=label_dict, tree_info=info, k=10, figsize=(18, 14),
)
plt.show()
print('High:', Counter(label_dict[y_train[i]] for i in top_idx).most_common())
print('Low: ', Counter(label_dict[y_train[i]] for i in bot_idx).most_common())

## 11. All-Node Activation Summary

In [ ]:
summary_df = analyze_all_nodes_summary(node_logits_train, y_train, label_dict, info)
print(summary_df.to_string())

## 12. Batch Export Node Analysis (metadata + images)

For each internal node, export to `./analysis/tree_nodes/IN_{idx}/`:
- `metadata.csv` \u2014 node statistics & per-class counts
- `top50/` \u2014 50 highest-activation images (if passing samples > 100)
- `bottom50/` \u2014 50 lowest-activation images (if passing samples > 100)

In [ ]:
def compute_internal_node_class_counts(leaf_class_counts, depth, num_classes):
    """Aggregate per-class counts upward from leaves to each internal node."""
    L = len(leaf_class_counts)
    N_int = L - 1
    counts = {i: np.zeros(num_classes, dtype=np.int64) for i in range(N_int)}
    for l in range(L):
        arr = leaf_class_counts[l]
        idx = l + N_int
        while idx > 0:
            parent = (idx - 1) // 2
            counts[parent] += arr
            idx = parent
    return counts


def get_samples_passing_node(node_idx, node_logits):
    """Return a boolean mask of samples that reach *node_idx* via hard routing."""
    N = node_logits.shape[0]
    if node_idx == 0:
        return np.ones(N, dtype=bool)
    path, cur = [], node_idx
    while cur > 0:
        parent = (cur - 1) // 2
        path.append((parent, cur == 2 * parent + 1))  # (parent, is_left)
        cur = parent
    path.reverse()
    mask = np.ones(N, dtype=bool)
    for p, go_left in path:
        lg = node_logits[:, p]
        mask &= (lg > 0) if go_left else (lg <= 0)
    return mask


def export_node_analysis(node_idx, node_logits, ds_images, y_labels, label_dict,
                         tree_info, train_internal_counts, internal_class_counts,
                         output_dir, k=50):
    """Export metadata.csv and top/bottom-K images for one internal node."""
    node_dir = Path(output_dir) / f'IN_{node_idx}'
    node_dir.mkdir(parents=True, exist_ok=True)

    ni = tree_info['internal_nodes'][node_idx]
    scores = node_logits[:, node_idx]
    class_counts = internal_class_counts[node_idx]

    passing = np.where(get_samples_passing_node(node_idx, node_logits))[0]

    if len(passing) > 0:
        ps = scores[passing]
        order = np.argsort(ps)
        ak = min(k, len(passing))
        top_k  = passing[order[::-1][:ak]]
        bot_k  = passing[order[:ak]]
        top_dom = Counter(y_labels[i] for i in top_k).most_common(1)[0]
        bot_dom = Counter(y_labels[i] for i in bot_k).most_common(1)[0]
        s_mean, s_std, s_max, s_min = ps.mean(), ps.std(), ps.max(), ps.min()
    else:
        top_k = bot_k = np.array([], dtype=np.int64)
        top_dom = bot_dom = (0, 0)
        s_mean = s_std = s_max = s_min = 0.0

    meta = {
        'node_idx': node_idx, 'layer': ni['layer'],
        'bias': f'{ni["b"]:.6f}',
        'score_mean': f'{s_mean:.6f}', 'score_std': f'{s_std:.6f}',
        'score_max': f'{s_max:.6f}',   'score_min': f'{s_min:.6f}',
        'num_passing': len(passing),
        'top50_dominant_label': label_dict[top_dom[0]],
        'top50_dominant_count': top_dom[1],
        'bottom50_dominant_label': label_dict[bot_dom[0]],
        'bottom50_dominant_count': bot_dom[1],
        'num': int(train_internal_counts[node_idx]),
    }
    for ci, cn in enumerate(label_list):
        meta[f'num_{ci}_{cn[:4]}'] = int(class_counts[ci])

    pd.DataFrame([meta]).to_csv(node_dir / 'metadata.csv', index=False)

    # Export images only when sufficient samples pass through
    if len(passing) > 100:
        for tag, indices in [('top50', top_k), ('bottom50', bot_k)]:
            d = node_dir / tag; d.mkdir(exist_ok=True)
            for rank, si in enumerate(indices):
                t = ds_images[si][0]
                arr = t.numpy().transpose(1, 2, 0) if isinstance(t, torch.Tensor) else np.array(t)
                arr = (arr * 255).astype(np.uint8) if arr.max() <= 1.0 else arr.astype(np.uint8)
                fname = f'{rank+1:02d}_{label_dict[y_labels[si]]}_{scores[si]:.4f}.png'
                Image.fromarray(arr).save(d / fname)
        print(f'  IN_{node_idx}: exported top50/bottom50')
    else:
        print(f'  IN_{node_idx}: passing={len(passing)} <= 100, skip images')

    return meta

In [ ]:
export_dir = Path('./analysis/tree_nodes')
export_dir.mkdir(parents=True, exist_ok=True)

num_classes = len(label_list)
internal_class_counts = compute_internal_node_class_counts(
    train_leaf_class_counts, tree.depth, num_classes)

all_metadata = []
for ni in range(tree.internal_node_num_):
    m = export_node_analysis(
        ni, node_logits_train, ds_train_img, y_train, label_dict,
        info, train_internal_counts, internal_class_counts, export_dir, k=50,
    )
    all_metadata.append(m)

all_meta_df = pd.DataFrame(all_metadata)
all_meta_df.to_csv(export_dir / 'all_nodes_metadata.csv', index=False)
print(f'\nDone. Exported {tree.internal_node_num_} nodes -> {export_dir / "all_nodes_metadata.csv"}')